# A guided tour of the 3DEXPERIENCE REST API

This notebook teaches the 3DX (ENOVIA) web API from scratch, using calls that are
**verified working** against tenant `R1132100093385`. Everything here is drawn from
`TENANT_API_FINDINGS.md` in this directory and from the working client at
`src/ViewpointGeneration/viewpoint_generation/viewpoint_generation/catalog/client.py`.

**Who this is for:** you know Python, but "REST API" is a phrase you have nodded
along to rather than used. No prior knowledge is assumed. Each section explains
the general web concept first, then shows the 3DX-specific call.

**How to use it:** fill in the settings cell (Section 2), then run cells top to
bottom. Read-only calls run freely. Calls that *write* to the live tenant are
switched off by default behind an `ALLOW_WRITES` flag — the tenant is a system of
record, and two orphaned test Documents from earlier experiments are still sitting
on it.

### Contents

| Section | What it covers |
|---|---|
| 1 | What a REST API actually is |
| 2 | Your settings — fill these in |
| 3 | The session and a few helpers |
| 4 | Logging in (your first `GET` and `POST`) |
| 5 | What every 3DX call needs |
| 6 | `GET` — reading data |
| 7 | `PUT` — asking for a ticket |
| 8 | `POST` — creating things |
| 9 | End to end: download a STEP, upload a plan |
| 10 | Reading errors |
| 11 | What this tenant does not have |

---
## 1. What a REST API actually is

When you open a web page, your browser sends an **HTTP request** to a server and
gets an **HTTP response** back. A REST API is the same machinery, except the
response is structured data (JSON) meant for a program instead of HTML meant for
a human.

Every request has four parts:

**1. A URL — the noun.** It names the thing you want.

```
https://…-space.3dexperience.3ds.com/enovia/resources/v1/modeler/dseng/dseng:EngItem/search
└──────────────── host ─────────────┘└─────────────────── path ──────────────────────────┘
```

The path is a hierarchy, read left to right: the `modeler` API, the `dseng`
(engineering) service, the `EngItem` resource, the `search` operation on it. 3DX
paths are unusually deep, but they are just folders-of-nouns.

**2. A method — the verb.** What you want done to that noun:

| Method | Means | Changes data? |
|---|---|---|
| `GET` | fetch it | no |
| `POST` | create something, or run an operation | yes |
| `PUT` | replace it, or hand the server a specific request | usually |
| `DELETE` | remove it | yes |

**3. Headers — metadata about the request.** Small `Name: value` pairs riding
alongside, saying who you are and what format you want back. 3DX needs three of
them; Section 5 covers which.

**4. A body — the payload.** Only for writes. For 3DX this is always JSON: a
Python dict that gets serialized and sent along.

The response comes back with a **status code** — a three-digit number — plus its
own headers and body:

| Code | Meaning | Typical cause here |
|---|---|---|
| `200` | OK | it worked… *usually*, see the trap below |
| `302` | Redirect | "go ask over there" — the login dance uses these |
| `400` | Bad request | your JSON or your `$mask` name was wrong |
| `401` / `403` | Unauthenticated / forbidden | bad or missing SecurityContext |
| `404` | Not found | that endpoint does not exist on this tenant |
| `500` | Server error | often a malformed search query |

> ### ⚠️ The single most important thing in this notebook
>
> **On 3DX, `200 OK` does not mean your write happened.** Several endpoints accept
> a request with the wrong method, answer `200` with an empty `data` array, and
> silently do nothing. Two real examples appear in Sections 7 and 8.
>
> Always judge success on **the object that came back**, never on the status code.

---
## 2. Your settings — fill these in

Everything the notebook needs is in this one cell.

**Where to find each value:**

- **Passport URL** — the host you log in against (the domain in your browser's
  address bar during login). On this tenant it is in the `eu1` region.
- **Space URL** — the 3DSpace host that holds the data, ending in `/enovia`. On
  this tenant it is in `usw2`. *These being different regions is normal for EDU
  tenants and is handled automatically — see Section 4.*
- **Tenant** — the platform id like `R1132100093385`, not a friendly name. Find it
  in any 3DX request's `tenant=` query parameter via browser DevTools.
- **Security context** — the `ctx::Role.Organization.CollabSpace` triplet that
  decides what you can see and do. Section 6.1 has a call that *lists yours for
  you*, so if you are unsure, put a guess here and fix it after running that cell.
- **Bookmark** — the name of the 3DX bookmark folder holding the parts you care
  about. Create one in the Bookmark Editor and drag items into it.

The same values live in this repo's `.env`, so the fallbacks below read from the
environment when it is set.

In [ ]:
import os

# --- Tenant endpoints -------------------------------------------------------
PASSPORT_URL = os.environ.get(
    "DX_PASSPORT_URL",
    "https://r1132100093385-eu1-academia.iam.3dexperience.3ds.com")

SPACE_URL = os.environ.get(
    "DX_SPACE_URL",
    "https://r1132100093385-usw2-academia-space.3dexperience.3ds.com/enovia")

TENANT = os.environ.get("DX_TENANT", "R1132100093385")

# --- Your login -------------------------------------------------------------
# This is the platform login id (e.g. CAN28), not your email address.
USERNAME = os.environ.get("DX_USERNAME", "CAN28")

# The ctx:: prefix is part of the value. Spaces in the org / space names are fine.
SECURITY_CONTEXT = os.environ.get(
    "DX_SECURITY_CONTEXT",
    "ctx::VPLMProjectLeader.Company Name.Colin Acton Space")

# --- What you want to look at ----------------------------------------------
BOOKMARK_NAME = os.environ.get("DX_BOOKMARK_SCOPE", "Inspection Parts")
COLLAB_SPACE = os.environ.get("DX_COLLAB_SPACE", "Colin Acton Space")

# --- Safety -----------------------------------------------------------------
# Sections 8 and 9 create real objects on a live PLM system. Flip this to True
# only when you actually intend to write, and clean up what you make.
ALLOW_WRITES = False

print(f"Passport : {PASSPORT_URL}")
print(f"Space    : {SPACE_URL}")
print(f"Tenant   : {TENANT}")
print(f"User     : {USERNAME}")
print(f"Context  : {SECURITY_CONTEXT}")
print(f"Bookmark : {BOOKMARK_NAME}")
print(f"Writes   : {'ENABLED' if ALLOW_WRITES else 'disabled (read-only)'}")

### The password

Never type a password into a notebook cell — it gets saved into the `.ipynb` file
and committed to git. `getpass` prompts for it instead and keeps it only in
memory.

In [ ]:
import getpass

PASSWORD = os.environ.get("DX_PASSWORD") or getpass.getpass(f"3DX password for {USERNAME}: ")
print(f"Password captured ({len(PASSWORD)} characters). It is not stored in this file.")

---
## 3. The session and a few helpers

`requests` is Python's standard library for HTTP. The key object is a
**`Session`**: it is a browser without a window. It remembers **cookies** across
calls, which is the whole reason logging in once lets later calls work — the
server hands back a cookie saying "this is Colin", and the session presents it on
every subsequent request.

The `show()` helper below just prints a response compactly so the notebook stays
readable. Look at what it prints: **method, URL, status code, then the body.**
That is the anatomy of every REST call.

In [ ]:
import json
import urllib.parse
from pathlib import Path

import requests

session = requests.Session()
session.headers.update({
    # 3DPassport's login form rejects some non-browser user agents.
    "User-Agent": "ViewpointGeneration-Catalog/1.0",
})


def show(response, label=None, limit=1500):
    """Print an HTTP response the way it is useful to read one."""
    if label:
        print(f"### {label}")
    print(f"{response.request.method} {response.url}")
    print(f"--> {response.status_code} {response.reason}   "
          f"({response.headers.get('content-type', 'no content-type')})")
    try:
        body = json.dumps(response.json(), indent=2)
    except ValueError:
        body = response.text
    print(body[:limit] + ("\n... [truncated]" if len(body) > limit else ""))
    print()


def brief(obj, limit=1500):
    """Pretty-print any Python object as JSON, truncated."""
    text = json.dumps(obj, indent=2, default=str)
    print(text[:limit] + ("\n... [truncated]" if len(text) > limit else ""))

---
## 4. Logging in — your first `GET` and `POST`

3DX does not use a simple API key. It uses **CAS** (Central Authentication
Service), a single-sign-on protocol. The idea: you prove who you are *once* to a
dedicated login server (the "passport"), and it issues tickets that other servers
(here, "3DSpace") accept as proof.

It takes three calls:

```
  1. GET  passport/login?action=get_auth_params   → a one-time login ticket (lt)
  2. POST passport/login  {lt, username, password} → sets a CASTGC cookie
  3. GET  space/…/CSRF                             → redirects via the passport,
                                                     comes back with ?ticket=ST-…,
                                                     and mints a space session
```

**Why three?** Step 1 gets a nonce so a stolen password POST cannot be replayed.
Step 2 authenticates you to the *passport only* — 3DSpace still does not know you.
Step 3 asks 3DSpace for something; 3DSpace bounces you to the passport, which says
"yes, this is Colin, here is a service ticket", and bounces you back. That final
hop is what gives you a 3DSpace session.

> **On the split regions:** the passport is in `eu1` and 3DSpace is in `usw2`.
> This looks alarming and is a non-issue — `requests.Session` follows the redirect
> chain automatically. Copying cookies between the two hosts by hand is neither
> necessary nor sufficient; 3DSpace will not issue a session without a service
> ticket.

### 4.1 `GET` the login ticket

A `GET` fetches something without changing anything. The `params=` argument
becomes the **query string** — the `?key=value&key=value` tail of the URL. Let
`requests` build it rather than gluing strings together; it handles escaping.

This exact call as `curl`, if you want to see the raw shape:

```bash
curl "$PASSPORT_URL/login?action=get_auth_params"
```

In [ ]:
response = session.get(
    f"{PASSPORT_URL}/login",
    params={"action": "get_auth_params"},
    timeout=60,
)
show(response, "Step 1 — request a login ticket")

login_ticket = response.json()["lt"]
print(f"Login ticket: {login_ticket[:40]}...")

### 4.2 `POST` your credentials

A `POST` sends data to the server. Here the body is **form-encoded** (the same
format an HTML `<form>` submits) rather than JSON, which is why it uses `data=`
instead of `json=`.

> ### ⚠️ Trap: a wrong password also returns `200`
>
> The passport does not answer a bad password with `401`. It re-renders the login
> page — which is a perfectly successful HTTP response. The real success signal is
> whether a **`CASTGC` cookie** appeared in the session.

In [ ]:
response = session.post(
    f"{PASSPORT_URL}/login",
    data={
        "lt": login_ticket,
        "username": USERNAME,
        "password": PASSWORD,
        "rememberMe": "false",
    },
    timeout=60,
)
print(f"Status code says: {response.status_code}  <- do NOT trust this alone")

cookies = [c.name for c in session.cookies]
authenticated = any(name.startswith("CASTGC") for name in cookies)

print(f"Cookies now held: {cookies}")
print()
print("Authenticated." if authenticated
      else "NOT authenticated — no CASTGC cookie. Check the username and password.")

### 4.3 `GET` the CSRF token (and the 3DSpace session)

A **CSRF token** is an anti-forgery measure: a random value the server hands you
and then demands back on every write, proving the write came from your session
and not from a malicious page that merely borrowed your cookies. 3DX calls its
header `ENO_CSRF_TOKEN`.

Fetching it does double duty — it is also the call that triggers the CAS redirect
chain and mints your 3DSpace session. Watch `response.history` below: those are
the `302` hops `requests` followed on your behalf.

In [ ]:
response = session.get(
    f"{SPACE_URL}/resources/v1/application/CSRF",
    params={"tenant": TENANT},
    timeout=60,
)

print("Redirect chain followed:")
for hop in response.history:
    print(f"  {hop.status_code} -> {hop.headers.get('location', '')[:95]}")
print(f"  {response.status_code} (final) {response.url[:95]}")
print()

CSRF_TOKEN = response.json()["csrf"]["value"]
print(f"CSRF token: {CSRF_TOKEN[:40]}...")
print(f"Session cookies now: {[c.name for c in session.cookies]}")

---
## 5. What every 3DX call needs

From here on, every data call carries the same three things. Rather than repeat
them, wrap them once — this is exactly what `DXClient._request()` does in the real
client.

| What | Where | Why |
|---|---|---|
| `SecurityContext` header | header | which role/org/space you are acting as. **Sent unencoded, `ctx::` prefix included.** |
| `ENO_CSRF_TOKEN` header | header | the anti-forgery token from 4.3 |
| `tenant=` | query string | required on cloud tenants |

> **Encoding gotcha:** the security context goes in the *header* unencoded, spaces
> and all. Only if you ever put it in a *query string* does it need percent-encoding
> (`ctx%3A%3AVPLM…`). Header and URL have different rules.

In [ ]:
def dx_headers(extra=None):
    """The headers every 3DX data call needs."""
    headers = {
        "Accept": "application/json",
        "SecurityContext": SECURITY_CONTEXT,   # unencoded, ctx:: included
        "ENO_CSRF_TOKEN": CSRF_TOKEN,
    }
    headers.update(extra or {})
    return headers


def dx_url(path):
    """Turn a /resources/... path into a full URL on the space host."""
    return f"{SPACE_URL.rstrip('/')}{path}"


def dx_request(method, path, params=None, headers=None, **kwargs):
    """Issue any request to 3DSpace with the standard headers and tenant param."""
    all_params = dict(params or {})
    all_params["tenant"] = TENANT
    return session.request(
        method,
        dx_url(path) if path.startswith("/") else path,
        params=all_params,
        headers=dx_headers(headers),
        timeout=60,
        **kwargs,
    )


def dx_get(path, params=None):
    return dx_request("GET", path, params=params)


def dx_post(path, body, params=None):
    return dx_request("POST", path, params=params, json=body,
                      headers={"Content-Type": "application/json"})


def dx_put(path, body, params=None):
    return dx_request("PUT", path, params=params, json=body,
                      headers={"Content-Type": "application/json"})


print("Helpers ready: dx_get, dx_post, dx_put")

### A second trap worth wiring in now: sessions expire invisibly

When your 3DX session times out, you do **not** get a `401`. You get redirected to
the passport, which returns its login page as HTML — with status `200`. Your code
then tries to parse a login form as JSON and reports a nonsense error.

This detector is worth keeping in any 3DX code you write.

In [ ]:
def looks_like_login_page(response):
    """True when a data call was quietly answered by the login page."""
    content_type = response.headers.get("content-type", "")
    if "text/html" in content_type and "login" in response.url.lower():
        return True
    return "action=get_auth_params" in response.text[:2000]


probe = dx_get("/resources/v1/application/CSRF")
print(f"Session healthy: {not looks_like_login_page(probe)}")

---
## 6. `GET` — reading data

`GET` is the read verb: it fetches a resource and changes nothing. Two ways to
tell the server *which* data you want:

- **Path parameters** — part of the URL itself: `…/dseng:EngItem/{id}`. Use when
  you know exactly which object you want.
- **Query parameters** — the `?key=value` tail: `…/search?$searchStr=bracket&$top=50`.
  Use for options: filtering, paging, how much detail.

3DX query parameters mostly start with `$`. The important ones:

| Parameter | Does |
|---|---|
| `$searchStr` | the search query |
| `$mask` | **how much detail** to return |
| `$top` | max results in this page |
| `$skip` | how many to skip (paging) |

### Masks — the concept that trips everyone

A **mask** is a named preset for "which fields do I want back". Instead of listing
fields, you name a shape: `dskern:Mask.Default`.

> ### ⚠️ Mask names are per-service and *not* consistent
>
> `dseng` (engineering items) accepts only `dskern:Mask.Default`. Bookmarks use a
> completely different family, `dsbks:BksMask.*`. Derived outputs use a third,
> `dsmvdo:DerivedOutputsMask.*`.
>
> A wrong mask returns **`400 Mask does not exist`** — which reads exactly like
> "this service cannot do that". It does not mean that. This cost days of work
> before someone read Dassault's own C# SDK
> ([3ds-cpe-emed/ws3dx-dotnet](https://github.com/3ds-cpe-emed/ws3dx-dotnet)) and
> found the real name. **Check the SDK before concluding a capability is missing.**

### 6.1 Who am I? — discovering your security context

If you were unsure what to put in `SECURITY_CONTEXT`, this answers it. It asks the
People & Organization service for your credentials and lists every
role/organization/space triplet you hold.

(The `e6wCurrentUser` endpoint that the setup checklist suggests for this **404s on
this tenant** — this is the working substitute.)

In [ ]:
person_path = "/resources/modeler/pno/person"

preferred = dx_get(person_path, {"current": "true", "select": "preferredcredentials"}).json()
credentials = preferred.get("preferredcredentials") or {}

if credentials:
    triplet = (f"ctx::{credentials.get('role', {}).get('name')}"
               f".{credentials.get('organization', {}).get('name')}"
               f".{credentials.get('collabspace', {}).get('name')}")
    print(f"Preferred context: {triplet}")

spaces = dx_get(person_path, {"current": "true", "select": "collabspaces"}).json()
print("\nAll contexts available to you:")
for collab_space in spaces.get("collabspaces", []) or []:
    for couple in collab_space.get("couples", []) or []:
        print(f"  ctx::{couple.get('role', {}).get('name')}"
              f".{couple.get('organization', {}).get('name')}"
              f".{collab_space.get('name')}")

### 6.2 Searching for parts

The workhorse read. `$searchStr` takes free text; `*` matches everything visible
to your security context.

In [ ]:
response = dx_get(
    "/resources/v1/modeler/dseng/dseng:EngItem/search",
    {"$searchStr": "*", "$mask": "dskern:Mask.Default", "$top": "5"},
)
show(response, "Search engineering items", limit=2000)

The interesting fields on each returned member:

| Field | Meaning |
|---|---|
| `id` | the object's unique hex id — you will pass this around constantly |
| `title` / `name` | display title and part number |
| `revision` | e.g. `A.1` |
| `cestamp` | a **change stamp** — a fingerprint of the object's current state |
| `state` | maturity: `IN_WORK`, `RELEASED`, … |
| `collabspace` | which collaborative space owns it |

`cestamp` is the useful one for syncing: if it is unchanged, nothing about the
object changed, so cached files are still valid. That is how the catalog avoids
re-downloading STEP files on every sync.

In [ ]:
members = response.json().get("member", [])
print(f"Got {len(members)} items\n")
for item in members:
    print(f"  {item.get('id')}  {item.get('title', '')[:34]:34}  "
          f"rev {str(item.get('revision')):5}  {str(item.get('state')):10}  "
          f"cestamp {str(item.get('cestamp'))[:12]}")

> ### ⚠️ Trap: `totalItems` is the *page size*, not the result count
>
> Ask for `$top=200` and the search reports `"totalItems": 200` — it is describing
> what it just handed you, not how many exist. Using it as a paging bound
> (`while skip < totalItems`) silently indexes only the first page.
>
> **Page until a short page arrives instead.** (Note that bookmarks, in 6.5, report
> a *genuine* total. It differs per service.)

In [ ]:
response_2 = dx_get(
    "/resources/v1/modeler/dseng/dseng:EngItem/search",
    {"$searchStr": "*", "$mask": "dskern:Mask.Default", "$top": "5", "$skip": "5"},
)
page_2 = response_2.json()

print(f"Page 1: totalItems = {response.json().get('totalItems')}, "
      f"members = {len(response.json().get('member', []))}")
print(f"Page 2: totalItems = {page_2.get('totalItems')}, "
      f"members = {len(page_2.get('member', []))}")
print("\nBoth 'totals' just echo the page size. Page until len(member) < $top.")

### 6.3 Server-side filtering with tag predicates

Free-text search is blunt. The backend also accepts **tag predicates** — structured
`field:value` filters — but **only in bracket form**:

- `ds6w:label:Test` → **HTTP 500**
- `[ds6w:label]:Test` → works

That one syntax detail is why an early pass concluded there was no structured
search at all. Confirmed tags on this tenant:

| Predicate | Filters on |
|---|---|
| `[ds6w:project]:"Colin Acton Space"` | collaborative space |
| `[ds6w:label]:Test` | object title |
| `[ds6w:type]:VPMReference` | object type |

They `AND` together and combine with free text. Filtering **on the server** matters:
a 300-item scan scoped this way returns 300 relevant items instead of ~19.

> **Careful:** an unrecognized tag returns `200` with **zero results**, not an
> error. "0 results" never proves a tag name is valid.

In [ ]:
queries = {
    "everything":        "*",
    "by space":          f'[ds6w:project]:"{COLLAB_SPACE}"',
    "by space AND text": f'[ds6w:project]:"{COLLAB_SPACE}" AND Test',
    "by type":           "[ds6w:type]:VPMReference",
}

for label, query in queries.items():
    result = dx_get(
        "/resources/v1/modeler/dseng/dseng:EngItem/search",
        {"$searchStr": query, "$mask": "dskern:Mask.Default", "$top": "20"},
    )
    count = len(result.json().get("member", [])) if result.status_code == 200 else "ERROR"
    print(f"  {label:20} {str(count):>6} items   {query}")

### 6.4 Fetching one object by id

When you know the id, put it in the **path** instead of searching. Note the
response still wraps the single object in a `member` list — 3DX is consistent
about that shape, which is convenient for parsing.

In [ ]:
ITEM_ID = members[0]["id"] if members else None
print(f"Using item: {ITEM_ID}\n")

if ITEM_ID:
    response = dx_get(
        f"/resources/v1/modeler/dseng/dseng:EngItem/{ITEM_ID}",
        {"$mask": "dskern:Mask.Default"},
    )
    show(response, f"Fetch item {ITEM_ID}", limit=1800)

### 6.5 Bookmarks — the curated scope

A **bookmark** in 3DX is a folder an operator drags objects into. It is the sanest
way to scope a catalog: instead of guessing which of 1000+ items matter, let a
human curate the list in the UI.

Two calls: find the bookmark by name, then list its contents.

> ### ⚠️ The mask trap in the flesh
>
> Bookmark contents are reachable via **`dsbks:BksMask.Items`**. The spelling every
> other service suggests — `dsbks:Mask.Items` — returns `400 Mask does not exist`,
> which looks exactly like "bookmark membership is not exposed by this API." It is
> exposed. The constant is just named differently. Valid masks here:
> `BksMask.Items`, `BksMask.Bookmarks`, `BksMask.Parent`, `BksMask.Linkable`.
> There is no `BksMask.Default`.

In [ ]:
response = dx_get(
    "/resources/v1/modeler/dsbks/dsbks:Bookmark/search",
    {"$searchStr": BOOKMARK_NAME, "$mask": "dskern:Mask.Default", "$top": "20"},
)
bookmarks = response.json().get("member", [])

print(f"Bookmarks matching {BOOKMARK_NAME!r}:")
for bookmark in bookmarks:
    print(f"  {bookmark.get('id')}  {bookmark.get('title')}")

bookmark_id = next(
    (b.get("id") for b in bookmarks
     if (b.get("title") or "").strip().lower() == BOOKMARK_NAME.strip().lower()),
    None,
)
print(f"\nExact match: {bookmark_id}")

In [ ]:
# The wrong mask, so you recognize the failure when you meet it.
if bookmark_id:
    wrong = dx_get(f"/resources/v1/modeler/dsbks/dsbks:Bookmark/{bookmark_id}",
                   {"$mask": "dsbks:Mask.Items", "$top": "10"})
    print(f"dsbks:Mask.Items    -> {wrong.status_code}  {wrong.text[:110]}")

    right = dx_get(f"/resources/v1/modeler/dsbks/dsbks:Bookmark/{bookmark_id}",
                   {"$mask": "dsbks:BksMask.Items", "$top": "1000", "$skip": "0"})
    print(f"dsbks:BksMask.Items -> {right.status_code}  (this is the one)")

In [ ]:
bookmark_items = []

if bookmark_id:
    payload = right.json()
    items = (payload.get("member") or [{}])[0].get("items") or {}

    for entry in items.get("member") or []:
        referenced = entry.get("referencedObject") or {}
        if referenced.get("identifier"):
            bookmark_items.append({
                "id": referenced["identifier"],
                "type": referenced.get("type"),
                "relative_path": referenced.get("relativePath") or "",
            })

    # Unlike search, THIS totalItems is genuine and can drive paging ($top caps at 1000).
    print(f"Bookmark reports totalItems = {items.get('totalItems')} (trustworthy here)")
    print(f"Collected {len(bookmark_items)} members\n")
    for item in bookmark_items[:10]:
        print(f"  {item['id']}  {str(item['type']):22}  {item['relative_path'][:52]}")

Bookmarks hold **whatever the operator dragged in** — this tenant's contain
`VPMReference`, `Document`, `Electrical3DSystem`, `Requirement Group`,
`System Scope`. Filter to engineering items before treating members as parts.

Sub-folders are separate objects listed under `BksMask.Bookmarks`; real code
recurses into them with a cycle guard (see `list_bookmark_items()` in `client.py`).

In [ ]:
def is_eng_item(item):
    """Bookmarks hold anything; keep only engineering items."""
    return ("dseng:EngItem" in (item.get("relative_path") or "")
            or item.get("type") == "VPMReference")


eng_items = [item for item in bookmark_items if is_eng_item(item)]
print(f"{len(eng_items)} of {len(bookmark_items)} bookmark members are engineering items")

if bookmark_id:
    sub = dx_get(f"/resources/v1/modeler/dsbks/dsbks:Bookmark/{bookmark_id}",
                 {"$mask": "dsbks:BksMask.Bookmarks", "$top": "1000"})
    children = ((sub.json().get("member") or [{}])[0].get("bookmarks") or {}).get("member") or []
    print(f"{len(children)} sub-folder(s) to recurse into")

### 6.6 A different view of the same object

3DX exposes the *same* object through several services, each showing different
fields. The `documents` service is the file-oriented view — it is where attached
files live, and it is the service the whole upload/download flow in Sections 7–9
runs through.

Two calls: object detail, and its file list.

In [ ]:
if ITEM_ID:
    detail = dx_get(f"/resources/v1/modeler/documents/{ITEM_ID}")
    entries = detail.json().get("data", []) or []
    if entries:
        elements = entries[0].get("dataelements", {}) or {}
        print("Fields from the documents view:")
        brief({key: elements.get(key) for key in
               ("title", "description", "type", "modified", "image", "typeicon")})

    files = dx_get(f"/resources/v1/modeler/documents/{ITEM_ID}/files")
    file_list = files.json().get("data", []) or []
    print(f"\nAttached files: {len(file_list)}")
    for entry in file_list:
        print(f"  {entry.get('id')}  {(entry.get('dataelements') or {}).get('title')}")

> **On `image` and `typeicon`:** they look like thumbnails and are not. Both point
> at `/snresources/images/icons/…` — **per-type** artwork, byte-identical for every
> Physical Product on the tenant. There is no per-object preview image here at all;
> the catalog renders its own previews locally from the STEP geometry instead.

---
## 7. `PUT` — asking for a ticket

`PUT` conventionally means "replace this resource with what I am sending". 3DX
uses it for something slightly different, and you cannot guess which endpoints
want it — you have to know.

**File transfer on 3DX is a two-server dance.** Bytes never move through 3DSpace.
Instead:

```
  1. Ask 3DSpace for a ticket        ← PUT, and only PUT
  2. Move the bytes to/from FCS      ← a completely separate host
  3. Tell 3DSpace it is done          ← POST, and only POST  (Section 8)
```

**FCS** (File Collaboration Server) is the file host. The **ticket** is a signed,
short-lived permission slip: it authorizes one transfer and tells FCS which store
and object the bytes belong to. FCS authenticates by ticket, not by your session —
so requests to FCS deliberately do *not* carry your 3DX headers.

Both ticket requests are `PUT`. A `GET` to the same URL returns an empty result
rather than an error.

### 7.1 `PUT` a download ticket

Note the body shape: 3DX wraps almost everything in `{"data": [ … ]}`, with the
real fields under `dataelements`. Getting used to that shape saves a lot of time.

In [ ]:
# Prefer a curated bookmark item; fall back to whatever the search turned up.
DOC_ID = eng_items[0]["id"] if eng_items else ITEM_ID

response = dx_put(
    f"/resources/v1/modeler/documents/{DOC_ID}/files/DownloadTicket",
    {"data": [{}]},          # empty selector = "whatever files this object has"
)
show(response, "PUT a download ticket", limit=1200)

ticket_url = ""
for entry in response.json().get("data", []) or []:
    elements = entry.get("dataelements", {}) or {}
    ticket_url = elements.get("ticketURL") or elements.get("ticketUrl") or ""
    if ticket_url:
        print(f"Ticket URL: {ticket_url[:110]}...")
        print(f"Filename  : {elements.get('title') or elements.get('fileName')}")
        break

if not ticket_url:
    print("No downloadable file on this object — expected for a bare EngItem.")

> **Expect `"The requested object contains no files."` here.** Engineering items on
> this tenant genuinely carry no attached files — on a `VPMReference` the geometry
> lives on a separate object, and the CAD conversion is published as a *derived
> output* instead. That is what Section 9.1 downloads. This ticket call is the
> right one for **Documents** (like uploaded inspection plans), which do have files.

### 7.2 `PUT` a check-in (upload) ticket

The mirror image, used before uploading. You declare how many files you intend to
send and get back the FCS endpoint plus a job ticket.

> ### ⚠️ Trap: this one really is `PUT`-only
>
> Send it as `POST` and you get an error. Meanwhile the *completion* call in 8.3
> is `POST`-only and answers a `PUT` with `200` and an empty result — doing
> nothing at all. Same flow, opposite verbs, and one of them fails silently.

In [ ]:
def get_checkin_ticket(doc_id, file_count=1):
    """PUT for an FCS upload ticket. Returns the dataelements dict, or None."""
    response = dx_put(
        f"/resources/v1/modeler/documents/{doc_id}/files/CheckinTicket",
        {"data": [{"dataelements": {"numberOfFiles": str(file_count)}}]},
    )
    if response.status_code >= 400:
        print(f"Ticket request failed ({response.status_code}): {response.text[:200]}")
        return None
    for entry in response.json().get("data", []) or []:
        elements = entry.get("dataelements", {}) or {}
        if elements.get("ticketURL") or elements.get("ticket"):
            return elements
    return None


print("Defined get_checkin_ticket(). It is exercised in Section 8.4, where a")
print("Document exists to check a file into.")

A ticket comes back looking like this. **Three fields, and the third is the one
people miss:**

```json
{
  "ticketURL":       "https://usw2-academia-dfcs.3dexperience.3ds.com/fcs/servlet/fcs/checkin",
  "ticketparamname": "__fcs__jobTicket",
  "ticket":          "QEBlbnZlbG9wM0BmY3NrZXlf…"
}
```

`ticketparamname` is **the name of the form field the ticket must be submitted
under**. Read it from the response rather than hard-coding `__fcs__jobTicket` —
Section 8.4 shows why it matters.

---
## 8. `POST` — creating things and running operations

`POST` covers two jobs:

1. **Create a new resource** — "here is a Document, make one".
2. **Run an operation** — a query too complex for a URL, or an action with side
   effects. 3DX uses `POST` for some pure *reads* for this reason (8.1 is one).

> **Everything from here writes to a live PLM system.** Cells are guarded by
> `ALLOW_WRITES` from Section 2. Leave it `False` to read the code without
> creating objects.

In [ ]:
def writes_enabled(what):
    """Guard for cells that modify the tenant."""
    if not ALLOW_WRITES:
        print(f"SKIPPED: {what}")
        print("Set ALLOW_WRITES = True in Section 2 to run this for real.")
        return False
    return True

### 8.1 `POST` as a read — locating derived outputs

A **derived output** is a file the platform generated from CAD by conversion — most
usefully a **STEP** file, the neutral CAD format the inspection pipeline can
actually load. Native SolidWorks/CATIA files are useless to it; STEP is the bridge.

This service is `POST`-only. There is no `GET` collection, and it demonstrates
three separate naming traps at once:

- the resource is `dsdo:DerivedOutput**s**` — plural; the singular `404`s
- the mask lives under a *third* prefix: `dsmvdo:DerivedOutputsMask.AllDetails`
- the body needs **both** `id` *and* `type` (omit `type` → `400 ReferencedObject
  must have ID and Type`)

> **Careful:** an item with *no* conversion answers `400`/`500 "Error in Get
> Derived Output Info"` rather than an empty list. Treat that as "none", not as a
> failure — otherwise one un-converted part fails an entire catalog sync.

In [ ]:
def list_derived_outputs(item_id, item_type="VPMReference"):
    """POST to /Locate to find files generated from an item's CAD."""
    body = {"referencedObject": [{
        "id": item_id,
        "type": item_type,                       # required, not optional
        "source": SPACE_URL.rstrip("/"),
        "relativePath": f"/resources/v1/modeler/dseng/dseng:EngItem/{item_id}",
    }]}

    response = dx_post(
        "/resources/v1/modeler/dsdo/dsdo:DerivedOutputs/Locate",
        body,
        params={"$mask": "dsmvdo:DerivedOutputsMask.AllDetails"},
    )
    if response.status_code >= 400:
        print(f"  (no derived outputs: {response.status_code} {response.text[:90]})")
        return []

    outputs = []
    for member in response.json().get("member") or []:
        derived = member.get("derivedOutputs") or {}
        if not derived.get("id"):
            continue
        files = derived.get("derivedOutputfiles") or derived.get("derivedoutputfiles") or []
        outputs.append({
            "id": derived["id"],
            "files": [{
                "id": entry.get("id"),
                "format": entry.get("format") or "",
                "filename": entry.get("filename") or "",
                "title": (entry.get("streamAttributes") or {}).get("title") or "",
                "filesize": entry.get("filesize"),
                "downloadable": entry.get("downloadable", True),
            } for entry in files],
        })
    return outputs


for item in (eng_items or [{"id": ITEM_ID}])[:3]:
    print(f"Item {item['id']}:")
    for output in list_derived_outputs(item["id"]):
        for entry in output["files"]:
            print(f"  {entry['format']:14} {entry['title'] or entry['filename']:30} "
                  f"{entry['filesize']} bytes")

### 8.2 `POST` to create — a Document

The straightforward case. Note the `{"data": [{"type": …, "dataelements": {…}}]}`
envelope again, and that the new object's id comes back in the response — that id
is the only handle you will have on it afterwards.

In [ ]:
import datetime

new_doc_id = None
doc_title = f"NOTEBOOK_DEMO_{datetime.datetime.now():%Y%m%dT%H%M%SZ}"

if writes_enabled(f"create a Document titled {doc_title}"):
    response = dx_post("/resources/v1/modeler/documents", {
        "data": [{
            "type": "Document",
            "dataelements": {
                "title": doc_title,
                "description": "Created by the 3DX REST API tour notebook.",
            },
        }],
    })
    show(response, "Create a Document", limit=1200)

    entries = response.json().get("data", []) or []
    # Judge success on the returned object, never on the status code.
    new_doc_id = entries[0].get("id") if entries else None
    print(f"New document id: {new_doc_id}" if new_doc_id
          else "No object came back — the create did NOT happen.")

### 8.3 The trap, demonstrated

The check-in completion endpoint is `POST`-only. Sent as `PUT`, the *identical*
payload returns:

```
200 OK      {"data": []}
```

Nothing was created. Nothing was reported. If you were checking `resp.ok`, you
would log a success and move on with an empty Document.

This is why every write in the real client returns `(object_or_None, message)`
based on the **returned object**.

In [ ]:
if new_doc_id and writes_enabled("demonstrate the PUT-vs-POST trap"):
    payload = {"data": [{"dataelements": {"title": "demo.txt", "receipt": "not-a-real-receipt"}}]}

    as_put = dx_put(f"/resources/v1/modeler/documents/{new_doc_id}/files", payload)
    print(f"As PUT : {as_put.status_code}  data = {as_put.json().get('data')!r}")
    print("         ^ looks fine, did nothing")

    as_post = dx_post(f"/resources/v1/modeler/documents/{new_doc_id}/files", payload)
    print(f"As POST: {as_post.status_code}  {as_post.text[:150]}")
    print("         ^ a real (here failing, since the receipt is fake) response")

### 8.4 `POST` to FCS — the multipart upload

Uploading a file is a different kind of `POST`: **multipart form data**, the
format an HTML form with a file picker sends. The body carries labelled parts —
here the file itself plus the job ticket.

> ### ⚠️ Trap: the file alone is not enough
>
> The job ticket must ride along **as a form field named by `ticketparamname`**.
> Post only the file and FCS accepts it at the HTTP level and stores nothing —
> your Document ends up with zero files.
>
> Note the asymmetry with *downloads* (Section 9), where the same kind of ticket
> goes in as a **query parameter** instead. Same ticket, two mechanisms.

This call goes to the **FCS host**, not 3DSpace, so it uses a bare `requests.post`
with no 3DX headers — FCS authenticates by ticket and rejects some of them.

In [ ]:
def upload_to_fcs(ticket, file_path):
    """POST one file to FCS. Returns the receipt string, or None."""
    file_path = Path(file_path)
    ticket_url = ticket.get("ticketURL")
    job_ticket = ticket.get("ticket")
    param_name = ticket.get("ticketparamname") or "__fcs__jobTicket"   # read it, do not assume

    with open(file_path, "rb") as handle:
        response = requests.post(
            ticket_url,
            data={param_name: job_ticket},                              # <-- the part people omit
            files={"file": (file_path.name, handle, "application/octet-stream")},
            timeout=300,
        )
    if response.status_code >= 400:
        print(f"FCS upload failed ({response.status_code}): {response.text[:200]}")
        return None
    return response.text.strip() or None


def complete_checkin(doc_id, receipt, file_name):
    """POST-only. Success is the returned file object, not the status code."""
    response = dx_post(f"/resources/v1/modeler/documents/{doc_id}/files", {
        "data": [{"dataelements": {"title": file_name, "receipt": receipt}}],
    })
    if response.status_code >= 400:
        return False, f"{response.status_code}: {response.text[:200]}"
    entries = response.json().get("data", []) or []
    if not entries:
        return False, "3DX accepted the request but registered no file."
    return True, f"Checked in as {entries[0].get('id')}"

In [ ]:
demo_file = Path("notebook_demo_payload.json")

if new_doc_id and writes_enabled(f"upload a file to document {new_doc_id}"):
    demo_file.write_text(json.dumps(
        {"note": "Uploaded by the 3DX REST API tour notebook.",
         "created": datetime.datetime.now().isoformat()}, indent=2))

    ticket = get_checkin_ticket(new_doc_id, file_count=1)
    if ticket:
        print(f"Ticket fields: {sorted(ticket)}")
        print(f"Form field name to use: {ticket.get('ticketparamname')!r}\n")

        receipt = upload_to_fcs(ticket, demo_file)
        print(f"FCS receipt: {str(receipt)[:80]}...\n" if receipt else "No receipt.\n")

        if receipt:
            ok, message = complete_checkin(new_doc_id, receipt, demo_file.name)
            print(f"Check-in {'succeeded' if ok else 'FAILED'}: {message}")

    demo_file.unlink(missing_ok=True)

### 8.5 Verify — then clean up after yourself

Two rules worth internalizing. **Read back what you wrote** (the status code
already proved it cannot be trusted). And **delete your test objects** — the
findings doc records two orphaned test Documents still on this tenant from earlier
work, because nobody cleaned up.

In [ ]:
if new_doc_id:
    files = dx_get(f"/resources/v1/modeler/documents/{new_doc_id}/files")
    attached = files.json().get("data", []) or []
    print(f"Document {new_doc_id} now has {len(attached)} file(s):")
    for entry in attached:
        elements = entry.get("dataelements") or {}
        print(f"  {entry.get('id')}  {elements.get('title')}  {elements.get('fileSize', '')}")
    print(f"\nRemember to delete {doc_title} ({new_doc_id}) in the 3DX UI when done.")

---
## 9. End to end

The two flows the catalog integration is actually built on, start to finish.

### 9.1 Download: 3DX → local STEP file

```
POST /Locate                      → find the STEP derived output
POST .../DownloadTicket           → get a signed FCS URL + job ticket
GET  {ticketURL}?ticket=…         → stream the bytes
```

> ### ⚠️ Trap: pass the job ticket as a real query *parameter*
>
> The ticket is base64 and contains `+` and `/` — characters with special meaning
> in a URL. Concatenate it into the URL string yourself and FCS answers
> `Failed to decrypt; FCS Bad ticket`. Hand it to `requests` as `params=` and let
> it percent-encode properly.
>
> Also note the `urllib.parse.quote(..., safe="")` on the file id in the path: ids
> can contain characters that would otherwise break the URL structure.

In [ ]:
def find_step_output(item_id):
    """Locate a downloadable STEP derived output. Returns (output_id, file)."""
    for output in list_derived_outputs(item_id):
        for entry in output["files"]:
            haystack = f"{entry.get('format', '')} {entry.get('filename', '')}".upper()
            if entry.get("downloadable") and ("STEP" in haystack or ".STP" in haystack):
                return output["id"], entry
    return None, None


def download_derived_output(output_id, file_id, dest):
    """Two-step FCS download: ticket, then stream the bytes."""
    path = (f"/resources/v1/modeler/dsdo/dsdo:DerivedOutputs/{output_id}"
            f"/dsdo:DerivedOutputFiles/{urllib.parse.quote(str(file_id), safe='')}"
            f"/DownloadTicket")

    response = dx_post(path, {})
    if response.status_code >= 400:
        return None, f"Ticket failed ({response.status_code}): {response.text[:180]}"

    elements = (response.json().get("data") or {}).get("dataelements") or {}
    ticket_url, job_ticket = elements.get("ticketURL"), elements.get("ticket")
    if not (ticket_url and job_ticket):
        return None, f"Incomplete ticket: {sorted(elements)}"

    param = elements.get("ticketparamname") or "__fcs__jobTicket"
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    # params=, NOT string concatenation — the ticket contains '+' and '/'.
    with session.get(ticket_url, params={param: job_ticket}, stream=True, timeout=300) as stream:
        if stream.status_code >= 400:
            return None, f"Download failed ({stream.status_code}): {stream.text[:180]}"
        with open(dest, "wb") as handle:
            for chunk in stream.iter_content(chunk_size=65536):
                if chunk:
                    handle.write(chunk)
    return dest, f"Downloaded {dest} ({dest.stat().st_size} bytes)"

In [ ]:
for item in (eng_items or [{"id": ITEM_ID}])[:5]:
    output_id, step_file = find_step_output(item["id"])
    if not output_id:
        continue

    print(f"STEP found on {item['id']}: {step_file['title'] or step_file['filename']} "
          f"({step_file['format']}, {step_file['filesize']} bytes)")

    path, message = download_derived_output(
        output_id, step_file["id"], Path("downloaded") / f"{item['id']}.stp")
    print(f"  {message}")

    if path:
        with open(path) as handle:
            first_line = handle.readline().strip()
        print(f"  First line: {first_line}   <- valid STEP starts with ISO-10303-21")
    break
else:
    print("No STEP derived output found on the items scanned.")

### 9.2 Upload: local plan → 3DX Document

Three calls chained, each a verb you have now seen:

```
POST /documents                              → create the Document       (8.2)
PUT  /documents/{id}/files/CheckinTicket     → get an upload ticket       (7.2)
POST {ticketURL}  multipart                  → move the bytes to FCS      (8.4)
POST /documents/{id}/files                   → register the file          (8.3)
```

If any step fails, say so plainly rather than reporting a success the operator
cannot use: a Document that exists but carries no file is worse than no Document.

In [ ]:
def upload_file_to_document(doc_id, file_path):
    """The full FCS check-in cycle for one file."""
    ticket = get_checkin_ticket(doc_id)
    if ticket is None:
        return False, "Could not get a check-in ticket."
    receipt = upload_to_fcs(ticket, file_path)
    if receipt is None:
        return False, "FCS upload produced no receipt."
    return complete_checkin(doc_id, receipt, Path(file_path).name)


def upload_plan(eng_item_id, plan_path, title):
    """Create a Document, check the plan file into it, and report honestly."""
    response = dx_post("/resources/v1/modeler/documents", {
        "data": [{"type": "Document", "dataelements": {
            "title": title,
            "description": "ViewpointGeneration inspection plan",
        }}],
    })
    entries = response.json().get("data", []) or [] if response.status_code < 400 else []
    doc_id = entries[0].get("id") if entries else None
    if not doc_id:
        return None, f"Document creation failed: {response.status_code} {response.text[:180]}"

    ok, message = upload_file_to_document(doc_id, plan_path)
    if not ok:
        return None, f"Created document {doc_id} but the file check-in failed: {message}"
    return doc_id, f"Uploaded inspection plan as document {doc_id}."


print("Defined upload_plan(). Set ALLOW_WRITES = True and call it with a real")
print("plan JSON to exercise it, e.g.:")
print('    upload_plan(ITEM_ID, "plan.json", "PLAN_part_A.1_ORDERED")')

> **What is missing from that chain:** in a complete design, a fourth call would
> *attach* the Document to its engineering item, so any cell could discover the
> plan from the part. **No document↔EngItem relationship endpoint exists on this
> tenant** — every documented route `404`s. The uploaded plan is a real Document
> with a real file, but it floats free; the link is recorded locally instead, so a
> cell re-finds its own plans and cross-cell discovery does not work yet.
>
> Plan titles carry the pipeline stage (`PLAN_<part>_<rev>_ORDERED`), and only the
> terminal `ordered` stage is uploaded by default.

---
## 10. Reading errors

The failure modes you will actually hit, and what each one means here.

| What you see | Usually means | Do |
|---|---|---|
| `200` + `{"data": []}` on a write | wrong verb (`PUT` where `POST` was needed) | check the returned object, flip the verb |
| `200` + HTML body | **session expired** — you got the login page | re-authenticate and retry |
| `400 Mask does not exist` | wrong mask *name*, not a missing capability | check the SDK for the real constant |
| `400 Payload is not valid` | body shape wrong — usually the `{"data": […]}` envelope | compare against a working call |
| `400 ReferencedObject must have ID and Type` | omitted `type` in a `dsdo` body | add it |
| `400/500 Error in Get Derived Output Info` | that item simply has no conversion | treat as "none", not an error |
| `404` | endpoint absent **on this tenant** | check the findings table before assuming it exists anywhere |
| `500` from search | malformed query — e.g. `ds6w:label:X` without brackets | use `[ds6w:label]:X` |
| `Failed to decrypt; FCS Bad ticket` | job ticket concatenated into the URL | pass it via `params=` |
| `403` everywhere | wrong `SecurityContext` | run Section 6.1 to list your real ones |

The single habit that catches most of these: **print the status code *and* the
first 200 characters of the body** on every failure. 3DX error messages are
genuinely informative — they are just easy to throw away.

In [ ]:
def diagnose(response):
    """Classify a 3DX response, including the failures that look like successes."""
    if looks_like_login_page(response):
        return "SESSION EXPIRED — got the login page as HTML with status 200."
    if response.status_code >= 400:
        # Collapse whitespace: error bodies are sometimes multi-line HTML.
        return f"HTTP {response.status_code}: {' '.join(response.text.split())[:150]}"
    try:
        payload = response.json()
    except ValueError:
        return f"Status {response.status_code} but the body is not JSON: {response.text[:120]}"

    rows = payload.get("data", payload.get("member"))
    if isinstance(rows, list) and not rows:
        return (f"HTTP {response.status_code} with an EMPTY result. On a write this "
                "usually means the wrong verb and NOTHING HAPPENED.")
    return f"HTTP {response.status_code}, {len(rows) if isinstance(rows, list) else 1} object(s)."


for label, response in [
    ("good search", dx_get("/resources/v1/modeler/dseng/dseng:EngItem/search",
                           {"$searchStr": "*", "$mask": "dskern:Mask.Default", "$top": "3"})),
    ("bad mask", dx_get("/resources/v1/modeler/dseng/dseng:EngItem/search",
                        {"$searchStr": "*", "$mask": "dskern:Mask.Details", "$top": "3"})),
    ("bad predicate", dx_get("/resources/v1/modeler/dseng/dseng:EngItem/search",
                             {"$searchStr": "ds6w:label:Test",
                              "$mask": "dskern:Mask.Default", "$top": "3"})),
    ("absent endpoint", dx_get("/resources/v1/application/e6w/api/v1/e6wCurrentUser")),
]:
    print(f"{label:18} {diagnose(response)}")

---
## 11. What this tenant does and does not have

Verified by live calls, not read from documentation.

### Works

| Capability | Endpoint | Verb |
|---|---|---|
| CSRF token | `/resources/v1/application/CSRF` | GET |
| Security contexts | `/resources/modeler/pno/person?current=true&select=…` | GET |
| EngItem search / fetch | `/resources/v1/modeler/dseng/dseng:EngItem[/search\|/{id}]` | GET |
| Object detail / files | `/resources/v1/modeler/documents/{id}[/files]` | GET |
| Bookmark list | `/resources/v1/modeler/dsbks/dsbks:Bookmark/search` | GET |
| Bookmark contents | same `/{id}`, `$mask=dsbks:BksMask.Items` | GET |
| Bookmark sub-folders | same `/{id}`, `$mask=dsbks:BksMask.Bookmarks` | GET |
| Library / class list | `/resources/v1/modeler/dslib/dslib:Library/search` | GET |
| CAD authoring file | `/resources/v1/modeler/dsxcad/dsxcad:Part/{id}` | GET |
| Derived outputs | `/resources/v1/modeler/dsdo/dsdo:DerivedOutputs/Locate` | **POST** |
| Derived output download | `…/dsdo:DerivedOutputFiles/{fileId}/DownloadTicket` | **POST** |
| Download ticket | `/resources/v1/modeler/documents/{id}/files/DownloadTicket` | **PUT** |
| Check-in ticket | `/resources/v1/modeler/documents/{id}/files/CheckinTicket` | **PUT** |
| FCS transfer | `{ticketURL}` (multipart) | **POST** |
| Check-in completion | `/resources/v1/modeler/documents/{id}/files` | **POST** |
| Document creation | `/resources/v1/modeler/documents` | **POST** |

### Does not work here

| Capability | Status |
|---|---|
| Per-object thumbnails / previews | **404** — only per-*type* icons; render locally instead |
| Representations (`dsrepr:`, `dsgeo:`, `dseng:EngRepInstance`) | **404 / empty** |
| Library / class members | **404** |
| Document↔EngItem relationship | **404** — the one real gap in the design |
| `e6wCurrentUser` | **404** — use the P&O route in 6.1 |
| Bookmark reverse lookup (`/dsbks:Bookmark/locate`) | `400 Payload is not valid` for every shape tried |
| Issue creation (`dsiss`) | untested |

> ### The lesson worth carrying forward
>
> **Consult the SDK before concluding a 3DX capability is absent.** This platform
> answers a wrong-but-well-formed mask with `400` and a wrong-but-well-formed path
> with `404` — *exactly* as it would if the feature did not exist. Bookmark
> membership was written off as unavailable through many rounds of probing; it
> worked the whole time, under a mask name no amount of guessing would reach.
>
> [**3ds-cpe-emed/ws3dx-dotnet**](https://github.com/3ds-cpe-emed/ws3dx-dotnet) is
> the best available reference for this platform's REST surface — Dassault's own
> C# SDK, with endpoints and mask names in the source comments. It covers `dseng`,
> `dsbks`, `dsdo`, `dsiss`, `dsmfg`, `dsprcs`, `dsxcad`. The official documentation
> portal needs a 3DEXPERIENCE ID, which is a separate identity from the tenant
> passport.

### Where to go next

- `TENANT_API_FINDINGS.md` — the full findings this notebook is built from
- `catalog/client.py` — the production version of every call here, with retry,
  re-login, and paging
- `catalog/sync.py` — how these calls compose into a catalog sync

---

**Before saving:** run *Kernel → Restart & Clear Output* if any cell output
captured tenant data you would rather not commit.